In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_11.pickle

In [ ]:
%%cudf.pandas.profile
### cell 11 ###

### cell 11 (optimized) ###

# Step 1: read all essays into Python lists
ids   = [p.rsplit('/', 1)[-1].replace('.txt', '') for p in tqdm(train_txt)]
texts = [open(p, 'r').read()                  for p in tqdm(train_txt)]

# Step 2: build a GPU‐backed DataFrame via pandas shim
# (we already have the cudf pandas extension loaded)
df_text = pd.DataFrame({'id': ids, 'text': texts})

# Step 3: vectorized string ops + cast to int64
df_text['essay_len']   = df_text['text'].str.strip().str.len().astype('int64')
df_text['essay_words'] = df_text['text'].str.count(r"\S+").astype('int64')

# Step 4: map stats back into `train` via a single GPU‐accelerated merge
# (avoids two .map() calls)
train = train.merge(
    df_text[['id', 'essay_len', 'essay_words']],
    on='id',
    how='left'
)
